# Clase 2.1: Parámetros de Generación y Ventana de Contexto
**ECI 2026 - Agentes de Inteligencia Artificial - UBA - Julio 2026**

¡Bienvenidos al taller práctico sobre Parámetros de Generación y Ventana de Contexto! En este notebooks nos enfocaremos en los controles técnicos que tenemos sobre las LLMs. Aprenderemos a ajustar la creatividad probabilística del modelo (Temperatura, Top-K, Top-P), a ponerle frenos estrictos (Stop Sequences) y a medir los límites de su memoria a corto plazo (Ventana de Contexto). Usaremos Google AI Studio (Gemini).

In [ ]:
# Instalamos las librerías necesarias
!pip install -q google-generativeai

In [ ]:
import os
import google.genai as genai
from google.genai import types
from google.colab import userdata

# Instrucciones:
# 1. Obtener API Key de Gemini en: https://aistudio.google.com/app/apikey
# 2. Guardar la clave en la sección 'Secrets' (🔑) de Colab con el nombre GEMINI_API_KEY.

gemini_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_key)
MODELO_GEMINI = 'gemini-2.5-flash'
print("Entorno configurado y cliente inicializado.")


## PARTE 1: Parámetros de Probabilidad
Controlan la distribución subyacente para elegir el siguiente token.
* **Temperatura (0.0 a 2.0+):** Controla la aleatoriedad. Un valor de `0` minimiza la aleatoriedad, haciendo las respuestas más predecibles. Valores altos aplanan las probabilidades haciendo respuestas más creativas pero propensas a mayor alucinaciones.
* **Top-K:** Obliga al modelo a elegir solo entre los *K* tokens más probables.
* **Top-P:** Elige a partir de un subconjunto dinámico de tokens cuya probabilidad acumulada alcanza el valor *P*.

In [ ]:
#Pueden ejecutarlo varias veces
#prompt_params = "Escribe el comienzo (solo un párrafo) de una historia de ciencia ficción sobre un relojero."
prompt_params = "Explica que es una lista"
print("--- TEMPERATURA 0.0 (Poca creatividad, analítico, repetitivo si se ejecuta varias veces) ---")
config_baja = types.GenerateContentConfig(
    temperature=0.0,
    top_p=0.8,
    top_k=10
)
resp_baja = client.models.generate_content(contents=prompt_params, model=MODELO_GEMINI, config=config_baja)
print(resp_baja.text, "\n")

print("--- TEMPERATURA 2.0 (Máxima creatividad, impredecible, léxico inusual) ---")
config_alta = types.GenerateContentConfig(
    temperature=2.0,
    top_p=0.99,
    top_k=40
)
resp_alta = client.models.generate_content(contents=prompt_params, model=MODELO_GEMINI, config=config_alta)
print(resp_alta.text)

## PARTE 2: Parámetros de Parada (Stop Sequences y Max Tokens)
Además de controlar *qué* elige el modelo, podemos controlar *cuándo se detiene*.
* **Max Output Tokens:** Un límite duro de cuántos tokens puede devolver. Útil para predecir costos y evitar que el modelo se explaye.
* **Stop Sequences:** Cadenas de texto que, si el modelo está a punto de generarlas, abortan la generación inmediatamente. Excelente para forzar al modelo a dar respuestas de un solo renglón o frenarlo al llenar formularios.

In [ ]:
prompt_lista = "Genera una lista de 10 lenguajes de programación muy antiguos:"

print("--- SIN STOP SEQUENCES ---")
resp_normal = client.models.generate_content(contents=prompt_lista, model=MODELO_GEMINI)
print(resp_normal.text, "\n")

print("--- CON STOP SEQUENCE (Cortar al llegar al punto 4) ---")
config_parada = types.GenerateContentConfig(
    stop_sequences=["4."],
    temperature=0.2
)
resp_parada = client.models.generate_content(contents=prompt_lista, model=MODELO_GEMINI, config=config_parada)
print(resp_parada.text)

## PARTE 3: Ventana de Contexto (Context Window)
La **ventana de contexto** es la "memoria a corto plazo" del modelo por cada interacción (entrada + salida). Se mide en tokens. Si se supera, el modelo "olvida" el principio del documento o arroja error. Gemini 2.5 Flash tiene una ventana monstruosa de 1 millón de tokens.

### Ejercicio Práctico: Conteo de Tokens Efectivo
Aprender a contar tokens antes de llamar a la API es una buena práctica de ingeniería para controlar costos y latencia.

In [ ]:
texto_ejemplo = "Los modelos de lenguaje avanzados de la familia Claude desarrollados por Anthropic, tales como Claude 3.5 Sonnet y Claude 3.7 Sonnet, destacan sobresalientemente por su extraordinaria e insuperable capacidad de razonamiento lógico, asistencia en programación y procesamiento masivo de información dentro de ventanas de contexto sumamente extensas."

resultado_tokens = client.models.count_tokens(
    model=MODELO_GEMINI,
    contents=texto_ejemplo
)

print(f"Texto original: '{texto_ejemplo}'")
print(f"Número de palabras: {len(texto_ejemplo.split())}")
print(f"Número total de TOKENS: {resultado_tokens.total_tokens}")
print(f"Ratio aproximado: {len(texto_ejemplo.split()) / resultado_tokens.total_tokens:.2f} palabras por token")


## PARTE 4: La Aguja en el Pajar (Needle in a Haystack)
Para testear que la ventana de contexto realmente procesa todo sin perder información (como pasaba con las arquitecturas antiguas), se usa la prueba *Needle in a Haystack*.

Consiste en inyectar un dato vital (la aguja) en un mar de texto irrelevante (el pajar).

In [ ]:
# 1. El Pajar
texto_relleno_inicio = "Log de servidor: Todo normal. CPU estable. Memoria OK. " * 1500
texto_relleno_fin = "Log de auditoría: Conexión cifrada establecida con éxito. " * 1500

# 2. La Aguja
aguja = "[ALERTA: El código de acceso de emergencia para el clúster es ECI-2026-XDF]"

# 3. El Documento Consolidado
documento_extenso = texto_relleno_inicio + aguja + texto_relleno_fin

tokens_pajar = client.models.count_tokens(model=MODELO_GEMINI, contents=documento_extenso).total_tokens
print(f"--- DIMENSIONES DEL EXPERIMENTO ---")
print(f"Tokens enviados en el prompt: {tokens_pajar}")

prompt_extraccion = f"""DOCUMENTO:
  {documento_extenso}

  PREGUNTA: ¿Se registró alguna alerta con un código de emergencia? Indícalo.
"""

print("Buscando la aguja en el pajar gigante...")
respuesta_contexto = client.models.generate_content(contents=prompt_extraccion, model=MODELO_GEMINI)
print("--- RESPUESTA ---")
print(respuesta_contexto.text)


### Consideraciones sobre 'Needle in a Haystack'

**Modelos más pequeños y antiguos (como algunos de Hugging Face):**

El test 'Needle in a Haystack' fue diseñado precisamente para exponer las limitaciones de los modelos más antiguos o con arquitecturas menos eficientes en el manejo de contextos largos. Es muy probable que un modelo más pequeño o uno con una ventana de contexto limitada (incluso si la ventana teórica es grande, pero la implementación es subóptima) tenga dificultades significativas para encontrar la 'aguja'.

*   **Rendimiento esperado:** Posiblemente fallen en identificar la información clave, respondan que no hay alerta, o alucinen una respuesta incorrecta, especialmente si la aguja está en medio de un contexto muy largo.
*   **Ventana de contexto real vs. teórica:** Algunos modelos pueden tener una ventana de contexto teórica grande, pero su capacidad para utilizar esa información de manera efectiva disminuye drásticamente a medida que la longitud del prompt aumenta. Este test ayuda a diferenciar entre la capacidad declarada y la capacidad de rendimiento real.

**Modelos modernos (como Qwen o Gemini):**

Los modelos de última generación, como Gemini 2.5 Flash (que estamos usando) y modelos como la serie Qwen de Alibaba Cloud, han sido diseñados con arquitecturas muy avanzadas para optimizar el rendimiento en ventanas de contexto extremadamente largas. Han mejorado significativamente en tareas de 'Needle in a Haystack' debido a:

*   **Atención optimizada:** Mecanismos de atención que escalan mejor con la longitud de la secuencia, permitiendo al modelo mantener el enfoque en todos los tokens.
*   **Entrenamiento:** Datos de entrenamiento que incluyen ejemplos con dependencias a largo alcance, preparando al modelo para este tipo de situaciones.
*   **Ventanas de contexto masivas:** La capacidad de procesar cientos de miles o incluso millones de tokens simultáneamente. El éxito del test en Gemini 2.5 Flash es un claro ejemplo de esto.

**¿Todos los modelos modernos resuelven el 'Needle in a Haystack'?**

Si bien la mayoría de los modelos de vanguardia demuestran un rendimiento excelente en esta prueba, decir que 'todos' la resuelven sería una generalización. El éxito aún depende de:

1.  **La arquitectura específica del modelo y su optimización.**
2.  **La calidad y diversidad de los datos de entrenamiento para contextos largos.**
3.  **La profundidad a la que se incrusta la aguja y la complejidad del 'pajar'.**

Sin embargo, la tendencia es clara: los modelos más recientes están diseñando para sobresalir en este tipo de desafíos, haciendo que la 'aguja en el pajar' sea una prueba cada vez más fácil para ellos, y menos un 'problema' real a medida que avanzan las investigaciones.